# EE200 Signals, Systems and Networks: Course Project
## Question 3: Sonic Signatures ('Magical Mystery Tune' & 'Zapptain America')

**Submitted By:**
1. K. Sumanth Reddy (Roll No. 240510)
2. Lella Sushanth (Roll No. 240595)

**Links:**
- Live Deployed App: [https://ee200-3rdq-hdmusyd23ieyrznjw26bch.streamlit.app](https://ee200-3rdq-hdmusyd23ieyrznjw26bch.streamlit.app)
- Source Code Repository: [https://github.com/kamjulasumanthred/EE200-3rdQ](https://github.com/kamjulasumanthred/EE200-3rdQ)

This notebook contains the complete implementation, visualizations, and robustness experiments for the audio fingerprinting system (inspired by Shazam).

### 1. Import Libraries and Load Modules

In [ ]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from audio_fingerprinter import (
    compute_spectrogram,
    get_peaks,
    generate_hashes,
    FingerprintDatabase,
    match_query,
    DEFAULT_FS
)

print("All libraries and modules loaded successfully!")

### 2. Spectrogram & Constellation Map Visualization
We load a sample Beatles song and plot:
1. Its spectrogram (in dB scale).
2. Its constellation map (detected local peaks).

In [ ]:
# Choose a sample song
sample_song = "Across The Universe"
song_path = f"EE200_course_project_data_2026/Q3_data/{sample_song}.mp3"

# Compute Spectrogram
spec_db, y, sr = compute_spectrogram(song_path)
print(f"Loaded '{sample_song}' with shape {y.shape} at {sr} Hz.")
print(f"Spectrogram shape (freq_bins, time_frames): {spec_db.shape}")

# Find local peaks (constellation)
peaks = get_peaks(spec_db)
print(f"Detected {len(peaks)} local peaks.")

# Plot Spectrogram (first 20 seconds)
plt.figure(figsize=(12, 5))
librosa.display.specshow(spec_db[:, :1000], sr=sr, hop_length=256, x_axis='time', y_axis='linear', cmap='magma')
plt.colorbar(format='%+2.0f dB')
plt.title(f"Spectrogram of '{sample_song}' (First 20s)")
plt.tight_layout()
plt.show()

# Plot Constellation Map (first 20 seconds)
plt.figure(figsize=(12, 5))
t_idx = [p[0] for p in peaks if p[0] < 1000]
f_idx = [p[1] for p in peaks if p[0] < 1000]
plt.scatter(t_idx, f_idx, color='cyan', s=15, edgecolors='black', linewidths=0.5)
plt.title(f"Constellation Map of '{sample_song}' Peaks (First 20s)")
plt.xlabel("Time (frames)")
plt.ylabel("Frequency bin index")
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3. Load Index Database
We load the database which has indexed all 50 songs.

In [ ]:
db = FingerprintDatabase()
db_path = "fingerprint_db.pkl"

if db.load(db_path):
    print(f"Successfully loaded database with {len(db.song_list)} indexed songs.")
else:
    print("Error: Run index_database.py first to compile fingerprint_db.pkl!")

### 4. Running a Query Match
We extract a 10-second slice from the middle of `Across The Universe` and test our database matching logic.

In [ ]:
# Slice a query clip
y_full, sr = librosa.load(song_path, sr=DEFAULT_FS)
start_sec = 45
duration_sec = 10
y_query = y_full[int(start_sec * sr) : int((start_sec + duration_sec) * sr)]

# Compute Spectrogram for query
stft_query = librosa.stft(y_query, n_fft=1024, hop_length=256)
spec_db_query = librosa.amplitude_to_db(np.abs(stft_query), ref=np.max)
peaks_query = get_peaks(spec_db_query)
hashes_query = generate_hashes(peaks_query)

print(f"Query has {len(peaks_query)} peaks and {len(hashes_query)} hashes.")

# Match against database
predicted_song, max_matches, offset_histograms, alignment_scores = match_query(hashes_query, db)

print(f"MATCH RESULT: Predicted '{predicted_song}' with {max_matches} matching hashes.")

# Plot Offset Histogram of the matched song
if predicted_song in offset_histograms:
    offsets = offset_histograms[predicted_song]
    plt.figure(figsize=(10, 4))
    plt.hist(offsets, bins=100, color='g', alpha=0.7, edgecolor='black')
    plt.title(f"Offset Histogram for Correct Match: '{predicted_song}'")
    plt.xlabel("Relative Time Offset (frames)")
    plt.ylabel("Match Count")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.show()

### 5. Comparative Analysis: Single Peaks vs. Paired Hashes
Let us evaluate why pairing peaks into hashes makes the system highly decisive compared to matching single peaks.

In [ ]:
# Execute experiment runner script which performs this comparison and plots the results
import subprocess
result = subprocess.run(["python", "run_experiments.py"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Error:", result.stderr)

### 6. Display Experiment Plots
Let's display the plotted results from our experiments.

In [ ]:
from PIL import Image

# 1. Single vs Paired Hashes comparison
if os.path.exists("plots/exp1_single_vs_pair.png"):
    display(Image.open("plots/exp1_single_vs_pair.png"))

# 2. Noise Robustness plot
if os.path.exists("plots/exp2_noise_robustness.png"):
    display(Image.open("plots/exp2_noise_robustness.png"))

### 7. Analytical Discussion

#### 1. Time vs. Frequency Resolution Trade-off
The Short-Time Fourier Transform (STFT) splits the signal into windowed segments. 
- A **short window** gives sharp time resolution (we know exactly *when* frequencies occur) but poor frequency resolution (frequencies are smeared across wide bins due to the uncertainty principle, where $\Delta t \cdot \Delta f \ge 1/4\pi$).
- A **long window** gives sharp frequency resolution (we can resolve nearby frequencies) but poor time resolution (we lose track of the exact timing within the window).

#### 2. Why Paired Hashes are Decisive compared to Single Peaks
If we match single peaks by frequency alone:
- A single peak is characterized only by one frequency, say $f_1$. Since many songs have similar frequencies (e.g., standard musical pitches like A=440Hz), a single query peak will match thousands of peaks in the database across almost all songs. This produces huge amounts of random background noise in the offset histogram.
- A **paired hash** combines two frequencies and their time difference: $(f_1, f_2, \Delta t)$. This three-dimensional feature space is highly specific. The probability of an accidental hash match is extremely low. Thus, incorrect songs get almost zero random matches (a flat histogram), while the correct song gets a very sharp peak (alignment) at the correct offset.

#### 3. Vulnerability to Pitch Shift and Time Stretch
- **Pitch Shift**: Pitch shifting shifts all frequencies by a multiplicative factor (or additive shift in log-frequency). This changes $f_1 \to f_1'$ and $f_2 \to f_2'$, which completely alters the hash keys. Since the hash keys $(f_1, f_2, \Delta t)$ rely on exact matches of frequency bins, even a small shift of 0.5 semitones destroys almost all matches.
- **Time Stretch**: Time stretching changes the duration of the audio, changing the time gap $\Delta t \to \alpha \Delta t$. Since the hash key relies on the exact frame difference $\Delta t$, this breaks hash matching. Furthermore, it shifts the relative offsets over time, meaning matches do not line up at a single constant offset, smearing the histogram peak.

#### 4. How to Improve Robustness against Pitch and Time Changes
- To handle pitch shift, we can use log-frequency spacing (constant-Q transform) where a pitch shift corresponds to a simple vertical translation. We can then match hashes based on relative frequency ratios (e.g. $f_2 / f_1$) instead of absolute frequency bins.
- To handle time stretch, we can define coordinates relative to robust time anchors (e.g., beat positions or tempo-normalized time) or extract scale-invariant descriptors.